# 📊 Phase 5: Final 3-Way Comparison Table

This notebook assembles your **portfolio centrepiece**:
- 3-way comparison: Base → SFT → DPO
- Visual chart of improvements
- Auto-generates README.md content

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

# Load all three metric files
with open('./data/baseline_metrics.json') as f:
    base = json.load(f)
with open('./data/sft_metrics.json') as f:
    sft = json.load(f)
with open('./data/dpo_metrics.json') as f:
    dpo = json.load(f)

metrics_order = ['json_validity_rate', 'exact_match_accuracy', 'avg_field_coverage']
labels = ['JSON Validity Rate', 'Exact Match Accuracy', 'Avg Field Coverage']
models = ['Base Model', 'SFT (QLoRA)', 'DPO (SFT+DPO)']
rows   = [base, sft, dpo]

print('\n' + '='*75)
print('📊 FINAL 3-WAY COMPARISON TABLE')
print('='*75)
header = f'{"Model":<22}' + ''.join(f'{l:>18}' for l in labels)
print(header)
print('-'*75)
for model_name, row in zip(models, rows):
    vals = ''.join(f'{row[m]:>18.1%}' for m in metrics_order)
    print(f'{model_name:<22}' + vals)
print('='*75)

print('\nDelta (Base → DPO):')
for m, l in zip(metrics_order, labels):
    delta = dpo[m] - base[m]
    print(f'  {l}: {delta:+.1%}')

In [ ]:
# ── Bar chart visualization ───────────────────────────────────────────────────
import numpy as np

x = np.arange(len(labels))
width = 0.25
colors = ['#94a3b8', '#3b82f6', '#10b981']

fig, ax = plt.subplots(figsize=(10, 6))

for i, (model_name, row, color) in enumerate(zip(models, rows, colors)):
    vals = [row[m] for m in metrics_order]
    bars = ax.bar(x + i * width, vals, width, label=model_name, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.0%}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Fine-Tuning Results: Base → SFT (QLoRA) → DPO', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('./data/comparison_chart.png', bbox_inches='tight')
plt.show()
print('✅ Chart saved to comparison_chart.png')

In [ ]:
# ── Generate README.md ────────────────────────────────────────────────────────
readme = f"""# Project 4: Fine-Tuning with LoRA & DPO
## Structured JSON Extraction — Qwen2.5-7B-Instruct

A fine-tuned language model for **structured JSON extraction from unstructured text**, 
using QLoRA for parameter-efficient supervised fine-tuning followed by DPO preference tuning.

---

## Results Summary

| Model | JSON Validity | Exact Match | Field Coverage |
|---|---|---|---|
| Base Model (Qwen2.5-7B-Instruct) | {base['json_validity_rate']:.1%} | {base['exact_match_accuracy']:.1%} | {base['avg_field_coverage']:.1%} |
| SFT Model (QLoRA fine-tuned) | {sft['json_validity_rate']:.1%} | {sft['exact_match_accuracy']:.1%} | {sft['avg_field_coverage']:.1%} |
| DPO Model (SFT + DPO) | {dpo['json_validity_rate']:.1%} | {dpo['exact_match_accuracy']:.1%} | {dpo['avg_field_coverage']:.1%} |

![Comparison Chart](data/comparison_chart.png)

---

## Task

**Structured JSON Extraction** — given messy, unstructured natural language text,
extract specified fields and return schema-compliant JSON. Four schema types:
person contact info, product listings, event details, and invoice data.

## Why Fine-Tune?

Even with careful prompt engineering, the base model frequently:
- Wraps JSON in markdown code fences (breaks parsers)
- Adds explanatory text before/after the JSON
- Misses or renames fields
- Returns inconsistent data types

Fine-tuning on 2,000+ task-specific examples eliminates these failure modes.

## Tech Stack

| Component | Tool |
|---|---|
| Base Model | Qwen2.5-7B-Instruct |
| Fine-Tuning | QLoRA (4-bit, rank=16) via HuggingFace TRL |
| Preference Tuning | DPO (beta=0.1) |
| GPU | Runpod A100 40GB |
| Tracking | Weights & Biases |

## Project Structure

```
00_setup.ipynb          # Environment setup & auth
01_data_prep.ipynb      # Dataset generation (SFT + DPO pairs)
02_baseline_eval.ipynb  # Base model evaluation (Row 1 of table)
03_sft_qlora.ipynb      # QLoRA fine-tuning (Row 2)
04_dpo_train.ipynb      # DPO preference tuning (Row 3)
05_final_comparison.ipynb  # This notebook — full comparison + charts
```

## What Went Wrong & How I Fixed It

> *(Fill this section in after training — this is what hiring managers want to see!)*

Examples of things to document:
- Did the model overfit early? (look for val_loss increasing while train_loss drops)
- Did it still add markdown fences sometimes? (document frequency and fix via data)
- Did boolean fields (`in_stock`) serialize inconsistently? (document and fix)
- Did DPO improve or slightly regress one metric? (explain the tradeoff)
"""

with open('./README.md', 'w') as f:
    f.write(readme)

print('✅ README.md generated!')
print('\n⚠️  Remember to fill in the "What Went Wrong" section with your actual observations!')